In [ ]:
import gondola as go
from glob import glob
import pandas as pd
import os
from tqdm import tqdm

#skip_indices = {2997,2999,3001}

print(go.__version__)

binary_path = '/home/gaps/tof-data/antarctica/nextcloud/flight_2025-26'
#binary_path = '/data1/nextcloud/cra_data/data/binaries_berkeley/starlink/'
#binary_path = '/home/gtytus/data/flight_bin_test/'

files = (sorted(glob(binary_path + '/*.bin')))

pos_moni_data = go.monitoring.SipPosMoniDataSeries()
pos_moni_data.max_size = int(10e6)
#t_1 = pos_moni_data.get_first_ts
#print(t_1)
print(pos_moni_data.__dir__())

rows = []

for idx, file in enumerate(tqdm(files)):
    pos_moni_data.add_telemetryfile(file)    
    
    t_1 = pos_moni_data.first_ts # gets the first monitoring time in the series
    #print(t_1)
    
    df = pos_moni_data.get_dataframe()

    for x in range(len(df)):
        rows.append({"timestamp": df["timestamp"][x], "altitude": df["altitude"][x]})


df_out = pd.DataFrame(rows)


print("nrows in rows =", len(rows))
print(df_out.columns)
print(df_out.head())


df_out.sort_values("timestamp", inplace=True)
df_out.to_hdf("altitude_vs_timeTEST.h5", key="altitude", mode="w")

In [ ]:
import gondola as gon
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import time

run_id = 251
paddles = gon.db.TofPaddle.all()



charge_a = []
charge_b = []
charge = []

data_path = "/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data"
dataset = Path(f"{data_path}/{run_id}")
files = list(dataset.glob("*.tof.gaps"))

print("nfiles =", len(files))

calib = gon.calibration.load_rb_calibrations(
    Path("/mnt/ucla-gaps-nas1/tof-data/antarctica/skua_hdd/data/calib/251123_215129UTC")
)
print("n calib entries =", len(calib))

# ---------------------------------
# loop over files / events
# ---------------------------------
endF = 5
for f in files[:endF]:  # or files[:endF]
    print("opening:", f)

    reader = gon.io.TofPacketReader(str(f))
    packcntEv = 0

    for pack in reader:
        # skip packets that are not actual TofEvent packets
        if pack.packet_type != gon.packets.TofPacketType.TofEvent:
            continue

        Qev = [0, 0, 0]
        numHits = [0, 0, 0]

        # IMPORTANT:
        # do NOT use ev.from_tofpacket(pack) here
        # for these files, the correct path is from_bytestream(pack.payload, 0)
        ev = gon.events.TofEvent.from_bytestream(pack.payload, 0)
        packcntEv += 1

        # optional slow-down for interactive debugging
        for rb in ev.rb_events:
            for hit in rb.hits:
                #print(hit.__dir__())
                #time.sleep(1)
                    #charge_a.append(hit.charge_a)
                #print(hit.paddle_id)
                #time.sleep(1)
                if int(hit.paddle_id) == 9:
                    charge.append((hit.charge_a + hit.charge_b)/2)




In [ ]:
plt.hist(charge, bins=100, range=(0, 500))
plt.yscale("log")
plt.xlabel("average charge on hit (pC)")
plt.ylabel("counts")
plt.title("charge distribution for paddle 9")
plt.show()

In [ ]:
go.__version__

In [ ]:
print("gondola version:", go.__version__)
print("gondola path:", go.__file__)
